# Code for parsing & merging experiment + questionnaire data from both sessions

## Imports

In [1]:
import contextlib
import json
import sqlite3
from collections.abc import Hashable, Iterable

import dateutil
import pandas as pd

from analysis_helpers.constants import (
    GOOGLE_FORM_DIR, 
    PROCESSED_DIR, 
    PSITURK_DIR
)

## Functions

In [2]:
def load_psiturk_data(db_filename: str) -> pd.DataFrame:
    # read "turkdemo" table from SQLite db, keep only necessary columns
    db_path = PSITURK_DIR.joinpath(db_filename)
    with contextlib.closing(sqlite3.connect(db_path)) as conn:
        df = pd.read_sql_query(
            'SELECT uniqueid, beginhit, status, datastring FROM turkdemo;', 
            conn
        )
        
    # drop rows with status code 1 (HIT was "Allocated" but experiment 
    # was never started) -- includes debugging runs & participant no-shows
    df = df.loc[df['status'] != 1]
    # all remaining rows should have status code 3 ("Completed")
    assert (df['status'] == 3).all()
    df = df.drop(columns='status')
    
    # drop first entry in db (full test run, not participant data)
    df = df.iloc[1:].reset_index(drop=True)
    
    # load psiTurk data JSON string
    df['datastring'] = df['datastring'].apply(json.loads)
    
    # replace "beginhit" timestamps with more accurate timestamps
    # ("beginhit" is load time of "Welcome" screen, loaded before 
    # participant arrived -- replace with load time of pre-experiment 
    # questionnaire)
    df['beginhit'] = df['datastring'].apply(
        lambda ds: ds['data'][0]['dateTime']
    )
    # updated timestamps should have the same relative order
    pd.testing.assert_index_equal(df.index, df.sort_values(by='beginhit').index)
    
    # add indicator column for testing room number
    df['testroom'] = int(db_path.stem.split('-')[-1])
    
    # convert to pandas nullable dtypes
    return df.astype({
        'uniqueid': 'string',
        'beginhit': 'Int64',
        'testroom': 'Int64'
    })

In [3]:
def load_questionnaire_data(filename: str) -> pd.DataFrame:
    df = pd.read_csv(GOOGLE_FORM_DIR.joinpath(filename))
    # convert datetime strings to POSIX timestamps (in ms) to match 
    # psiTurk data. Google Forms converts all dates to current local 
    # time when downloading data, so participants collected during EDT 
    # show EST equivalents. Pandas's default `date_parser` doesn't 
    # handle this behavior well, so use dateutil instead.
    df['Timestamp'] = (
        df['Timestamp']
        .apply(
            lambda ts_str: dateutil.parser.parse(
                ts_str, tzinfos={'EST': -18000}
            ).timestamp() 
            * 1000
        ).astype('Int64')
    )
    # drop test runs (where Subject ID is NaN)
    df = df.dropna(subset=['Subject ID']).reset_index(drop=True)
    # convert to pandas nullable types
    return df.convert_dtypes()

In [4]:
def merge_with_duplicates(
    left: pd.DataFrame, 
    right: pd.DataFrame, 
    on: Hashable | Iterable[Hashable], 
    **merge_kwargs
) -> pd.DataFrame:
    """
    Performs a 1:1 merge between two pandas DataFrames on columns that 
    may contain non-unique values. Rows from `left` and `right` 
    containing duplicate values in the merge column(s) are matched by 
    order of appearance (i.e., cumulative count of occurrences) and 
    non-unique row order is preserved in the merged result. Thus for the 
    result to be a proper 1:1 merge, each value in the merge colulmn(s) 
    must appear in `left` and `right` the same number of times.
    
    Parameters
    ----------
    left, right : pandas.DataFrame
        DataFrames to merge
    on : label or iterable of labels
        Column name(s) to merge on
    **merge_kwargs
        Additional keyword arguments passed to `pandas.DataFrame.merge`
    
    Returns
    -------
    pandas.DataFrame
        Merged DataFrame object
        
    """
    if not left[on].value_counts().sort_index().equals(
        right[on].value_counts().sort_index()
    ):
        raise ValueError(
            "'left' and 'right' must contain the same number of "
            f"occurrences of each value in {on}"
        )
        
    _left, _right = left.copy(), right.copy()
    _left['_'] = _left.groupby(on).cumcount()
    _right['_'] = _right.groupby(on).cumcount()
    
    if isinstance(on, str):
        _on = [on, '_']
    else:
        _on = list(on) + ['_']
    
    return _left.merge(_right, on=_on, **merge_kwargs).drop(columns='_')

## Participant data to exclude from analyses

In [5]:
# ================== data collection stopped mid-task ==================
# Task & post-questionnaire data not recorded. Drop pre-questionnaire 
# data before merging
errors_session1 = [
    'MD-1011318-A-05',   # system backup during experiment caused crash
    'MD-020119-A-01',    # participant accidentally pressed key to exit
    'MD-020119-B-01'     # participant felt ill and chose to drop out
]

# (drop ses. 2 pre-questionnaire before merging, ses. 1 data after)
errors_session2 = [
    'MD-101218-B-04'     # Docker daemon crashed during session 2 task
]

# ================= data dropped after task completion =================
# Pre-questionnaire, task, & post-questionnaire data recorded. Drop all 
# data ater merging.
dropids = errors_session2 + [
    'MD-020119-B-03',    # participant did not return for session 2
    'MD-102218-B-06',    # participant did not return for session 2
    'MD-101318-A-01',    # participant did not return for session 2
    'MD-013119-A-01',    # participant did not return for session 2
    'MD-102218-B-04',    # participant did not return for session 2
    'MD-102318-A-01',    # participant did not return for session 2
    'MD-022019-B-01',    # participant did not return for session 2
    'MD-020719-B-01',    # reported speakers cutting out during task
    'MD-102218-A-05',    # mic stopped recording stopped during recall
    'MD-102218-A-04',    # self-reported task difficulty due to migraine
    'MD-101318-A-04',    # did not follow task instructions
    'MD-101618-A-04',    # did not follow task instructions
    'MD-101618-B-03',    # did not follow task instructions
    'MD-102218-B-07',    # did not follow task instructions
]

## Load psiTurk experiment data

In [6]:
# load psiTurk data from both testing rooms
room1_df = load_psiturk_data('test-room-1.db')
room2_df = load_psiturk_data('test-room-2.db')

# concatenate testing room dataframes, order by experiment start time
psiturk_df = (
    pd.concat((room1_df, room2_df))
    .sort_values(by='beginhit')
    .reset_index(drop=True)
)
psiturk_df.head()

,uniqueid,beginhit,datastring,testroom
0,debugIEH2T:debugDLVLJ,1539368162836,"{'condition': 0, 'counterbalance': 0, 'assignm...",1
1,debugBUnNA:debugLtZcs,1539371956776,"{'condition': 0, 'counterbalance': 0, 'assignm...",1
2,debugYQfMB:debugxg7il,1539372566510,"{'condition': 0, 'counterbalance': 0, 'assignm...",2
3,debugd1YD1:debug4FrAg,1539375821845,"{'condition': 0, 'counterbalance': 0, 'assignm...",1
4,debug92cgv:debugvdAIT,1539376317256,"{'condition': 0, 'counterbalance': 0, 'assignm...",2


## Load pre- & post-experiment questionnaire responses

In [7]:
# load in responses, convert timestamps, drop test runs
preq_df = load_questionnaire_data('pre-experiment-questionnaire.csv')
postq_df = load_questionnaire_data('post-experiment-questionnaire.csv')

# # assign timestamp columns different names so both remain after merging
preq_df = preq_df.rename(columns={'Timestamp':'preqtime'})
postq_df = postq_df.rename(columns={'Timestamp':'postqtime'})

# remove pre-questionnaire data from participants with no corresponding
# task or post-questionnaire data (did not complete experiment session)
preq_df = preq_df.loc[
    # session 1
    ~preq_df['Subject ID'].isin(errors_session1) &
    # session 2
    ~preq_df.index.isin(
        preq_df.loc[preq_df['Subject ID'].isin(errors_session2)]
        .drop_duplicates('Subject ID', keep='last')
        .index
    )
].reset_index(drop=True)

## Merge task & questionnaire data

In [8]:
# all three dataframes should now have the same number of entries
assert psiturk_df.shape[0] == preq_df.shape[0] == postq_df.shape[0]

experiment_df = pd.concat((psiturk_df, preq_df), axis=1)
experiment_df = merge_with_duplicates(experiment_df, postq_df, on='Subject ID')

# correct a few typos in participant IDs in the Google Forms
experiment_df = experiment_df.replace({
    'MD--22819-B-01' : 'MD-022819-B-01',
    'MD-101218-A-06' : 'MD-102218-A-06',
    'MD-102118-B-06' : 'MD-102218-B-06',
    'MD-011319-A-02' : 'MD-013119-A-02'
})

# fix a repeated participant ID
experiment_df.loc[
    experiment_df.index[experiment_df['Subject ID'] == 'MD-101218-A-03'][2], 
    'Subject ID'
] = 'MD-102018-A-03'

# drop excluded participants
experiment_df = experiment_df.loc[
    ~experiment_df['Subject ID'].isin(dropids)
].reset_index(drop=True)

# all participant IDs should now appear in dataset exactly twice
assert (experiment_df['Subject ID'].value_counts() == 2).all()

experiment_df.head()

,uniqueid,beginhit,datastring,testroom,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",Is English your first language?,Do you have any hearing or speech impairments?,Do you have normal color vision?,...,What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?,postqtime,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,debugIEH2T:debugDLVLJ,1539368162836,"{'condition': 0, 'counterbalance': 0, 'assignm...",1,1539368143000,MD-101218-A-01,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,1.0,A little alert,1539371520000,Very engaging,Somewhat easy,Very well,Very well,A little tired
1,debugBUnNA:debugLtZcs,1539371956776,"{'condition': 0, 'counterbalance': 0, 'assignm...",1,1539371933000,MD-101218-B-01,I've never watched either one,Yes,No,Yes,...,undeclared,5.0,0.0,A little sluggish,1539374995000,A little engaging,Somewhat easy,Very well,Somewhat well,A little tired
2,debugYQfMB:debugxg7il,1539372566510,"{'condition': 0, 'counterbalance': 0, 'assignm...",2,1539372538000,MD-101218-A-02,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,2.0,A little alert,1539375570000,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired
3,debugd1YD1:debug4FrAg,1539375821845,"{'condition': 0, 'counterbalance': 0, 'assignm...",1,1539375802000,MD-101218-B-02,I've never watched either one,Yes,No,Yes,...,neuroscience,9.0,0.0,A little alert,1539378568000,Very engaging,Very easy,Very well,Somewhat well,A little alert
4,debug92cgv:debugvdAIT,1539376317256,"{'condition': 0, 'counterbalance': 0, 'assignm...",2,1539376286000,MD-101218-A-03,I've never watched either one,Yes,No,Yes,...,Sociology,7.0,0.0,A little alert,1539379114000,Very engaging,Somewhat easy,Very well,Somewhat well,Very alert


## Format & save across-session ID mapping for use in analyses

In [9]:
subid_mapping = (
    experiment_df.groupby("Subject ID", sort=False)["uniqueid"]
    .apply(lambda x: pd.Series(x.values, index=["session 1", "session 2"]))
    .unstack()
)

subid_mapping.head()

,session 1,session 2
Subject ID,,
MD-101218-A-01,debugIEH2T:debugDLVLJ,debug2Ea7T:debugosNZ7
MD-101218-B-01,debugBUnNA:debugLtZcs,debugQEynG:debugpwxCU
MD-101218-A-02,debugYQfMB:debugxg7il,debugyBEnU:debugSXeyx
MD-101218-B-02,debugd1YD1:debug4FrAg,debugbu5Bq:debugl91xD
MD-101218-A-03,debug92cgv:debugvdAIT,debugIFCgX:debugt0bgV


In [10]:
# subid_mapping.to_csv(PROCESSED_DIR.joinpath('subid-mapping.csv'))
# experiment_df.to_csv(PROCESSED_DIR.joinpath('questionnaire-data.csv'), index=False)